# Check the effective governing equation before MLP/PINN

This notebook fits no neural network. It checks two **assumed effective response laws**
using only the fixed **36 training cases**. The other 37 cases remain excluded.

Candidate: `Svm = R + A exp(-k (E11/0.1)^n)`; simple version fixes n=1.
The decreasing branch and E11 coordinate are our adaptation of an exponential-power
constitutive family, not a recovered source law or established pore mechanism.
Reference: https://doi.org/10.1016/0022-5096(75)90018-6
VPSC definition lead: https://github.com/lanl/VPSC_code

Upload the professor's existing ZIP to Google Drive. Set its path below.
The extractor reads only the matrix and 36 whitelisted histories from that ZIP.
All results are saved to a new Drive folder. Porosity 4% stays assumed metadata.
The final test partition has historical exposure; it is not a new blind test.

Run the cells in order. Hosted Colab execution has not been verified by the author;
the embedded checker and its mathematical/isolation tests were run locally.


In [ ]:
%pip install -q numpy==2.2.6 scipy==1.15.3 scikit-learn==1.7.2 matplotlib==3.10.9


Restart the runtime if Colab requests it after installing, then continue here.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
ZIP_SOURCE = Path('/content/drive/MyDrive/MMM-ML/TEAM_8_extracted_results_set_1.zip')
LOCAL_ROOT = Path('/content/mmm_equation_check')
OUTPUT = Path('/content/drive/MyDrive/MMM-ML/equation_checks') / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')


## Reproduce the frozen checker and case split

Scalers and coefficients are fitted within each fitting fold. Three case folds,
three initializations and alpha values 0.0001, 0.01, 1 are compared. Selection uses
development CV, so these scores are not an unbiased final accuracy estimate.
Condition-group holdouts are additional diagnostics using the selected alpha.
Screening limits: mean case-normalized RMSE <5%, worst <10%, improvement over
the fitting-mean-curve reference, optimizer convergence, and numerical checks.
These research screening limits were fixed before this qualification run.


In [ ]:
import json, sys, zipfile, hashlib
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
for name, content in {'configs/effective_equation.json': {'seed': 42, 'folds': 3, 'alphas': [0.0001, 0.01, 1.0], 'starts': 3, 'max_nfev': 1500, 'strain_min': 0.0002, 'strain_max': 0.0999, 'strain_points': 120, 'strain_reference': 0.1, 'families': ['exponential', 'power_exponential'], 'screening_mean_nrmse': 0.05, 'screening_worst_nrmse': 0.1, 'porosity_fraction_assumed': 0.04, 'test_targets_permitted': False}, 'data/split.json': {'seed': 42, 'train': [2, 3, 5, 6, 8, 18, 19, 21, 22, 25, 26, 27, 28, 29, 30, 33, 34, 38, 40, 41, 43, 46, 49, 52, 53, 60, 64, 65, 66, 69, 70, 72, 76, 78, 80, 81], 'test': [1, 4, 7, 9, 10, 11, 12, 13, 14, 15, 16, 17, 20, 23, 24, 31, 32, 35, 36, 37, 39, 42, 44, 45, 47, 48, 50, 51, 54, 67, 68, 71, 73, 74, 75, 77, 79]}}.items():
    path = LOCAL_ROOT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and json.loads(path.read_text()) != content:
        raise ValueError(f'Existing configuration differs: {path}')
    path.write_text(json.dumps(content, indent=2)+'\n')
source = '"""Training-case-only qualification of effective scalar response hypotheses.\n\nSvm = R + A exp(-k x**n), x=E11/0.1. These are softening adaptations,\nnot identified microscopic laws; E11 is an operational loading coordinate.\nThis module deliberately does not call data.build (which opens all cases).\n"""\nimport os\nos.environ.setdefault("OPENBLAS_NUM_THREADS", "1")\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport platform\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport scipy\nfrom scipy.integrate import solve_ivp\nfrom scipy.optimize import least_squares\nfrom scipy.special import expit, logit\nfrom sklearn.model_selection import KFold\n\nROOT = Path(__file__).resolve().parents[1]\n\n\ndef digest(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef load_training(raw, split_path, grid, expected_train=36, expected_test=37):\n    """Open only paths explicitly whitelisted by split.train; audit each source."""\n    raw = Path(raw)\n    split = json.loads(Path(split_path).read_text())\n    ids, test = split[\'train\'], split[\'test\']\n    if (len(ids) != expected_train or len(test) != expected_test or\n            len(set(ids)) != len(ids) or len(set(test)) != len(test) or set(ids) & set(test)):\n        raise ValueError(\'Invalid/disjoint training split\')\n    with (raw / \'simulation_matrix.csv\').open() as f:\n        matrix = {int(r[\'Simulation_ID\']): r for r in csv.DictReader(f)}\n    X, Y, audits = [], [], []\n    for case in ids:\n        path = raw / f\'simulation_{case:04d}\' / \'STR_STR.OUT\'\n        columns = \'TIME Evm Svm E11 E22 E33 E23 E13 E12 SCAU11 SCAU22 SCAU33 SCAU23 SCAU13 SCAU12 TEMP\'.split()\n        with path.open() as f:\n            if f.readline().split() != columns:\n                raise ValueError(f\'Unexpected source columns: {case}\')\n        a = np.loadtxt(path, skiprows=1)\n        if a.shape[1] != 16 or not np.isfinite(a).all() or not np.all(np.diff(a[:, 3]) > 0):\n            raise ValueError(f\'Invalid history: {case}\')\n        if grid[0] < a[0, 3] or grid[-1] > a[-1, 3]:\n            raise ValueError(f\'Extrapolation requested: {case}\')\n        r = matrix[case]\n        X.append([float(r[\'temperature\']), np.log10(float(r[\'strain_rate\'])),\n                  float(r[\'density\']), float(r[\'shear_modulus\'])])\n        Y.append(np.interp(grid, a[:, 3], a[:, 2]))\n        e, s = a[:, 3:9], a[:, 9:15]\n        vm = np.sqrt(.5 * ((s[:, 0]-s[:, 1])**2 + (s[:, 1]-s[:, 2])**2\n                          + (s[:, 2]-s[:, 0])**2) + 3*(s[:, 3:]**2).sum(1))\n        de = np.diff(e, axis=0)\n        dw = (s[1:, :3]*de[:, :3]).sum(1) + 2*(s[1:, 3:]*de[:, 3:]).sum(1)\n        reconstructed = a[0, 1] + np.cumsum(dw/a[1:, 2])\n        audits.append(dict(id=case, rows=len(a), sha256=digest(path),\n                           svm_relative_error=float(np.max(np.abs(vm-a[:, 2])/a[:, 2])),\n                           evm_work_reconstruction_max_error=float(np.max(np.abs(reconstructed-a[1:, 1]))),\n                           evm_reversals=int(np.sum(np.diff(a[:, 1]) < 0)),\n                           negative_work_increments=int(np.sum(dw < -1e-10)),\n                           temperature_offset=float(np.median(a[:, 15])-float(r[\'temperature\']))))\n    return np.array(ids), np.array(X), np.array(Y), audits, split\n\n\ndef basis(X, mean, scale):\n    z = (np.asarray(X)-mean)/scale\n    return np.column_stack([np.ones(len(z)), z, z[:, 0]*z[:, 1]])\n\n\ndef response(beta, B, strain, family, stress_scale=1., reference=.1, jacobian=False):\n    """Positive log-linear R,A,k; n in (0.1,8) by logistic mapping."""\n    count = 3 if family == \'exponential\' else 4\n    beta = np.asarray(beta).reshape(count, B.shape[1])\n    q = B @ beta.T\n    R, A, k = np.exp(q[:, :3]).T\n    n = np.ones(len(B)) if count == 3 else .1+7.9*expit(q[:, 3])\n    u = np.asarray(strain)/reference\n    if np.any(u <= 0):\n        raise ValueError(\'Positive strain required; zero is outside qualified domain\')\n    power = u[None, :]**n[:, None]\n    z = k[:, None]*power\n    damp = np.exp(-np.minimum(z, 700.))\n    excess = A[:, None]*damp\n    pred = (R[:, None]+excess)*stress_scale\n    if not jacobian:\n        return pred\n    dz = np.where(z < 700., z, 0.)\n    factors = [np.broadcast_to(R[:, None], z.shape), excess, -excess*dz]\n    if count == 4:\n        dn = 7.9*expit(q[:, 3])*(1-expit(q[:, 3]))\n        factors.append(-excess*dz*np.log(u)[None, :]*dn[:, None])\n    J = np.concatenate([(v[:, :, None]*B[:, None, :]).reshape(-1, B.shape[1])\n                        for v in factors], axis=1)*stress_scale\n    return pred, J\n\n\ndef fit_global(X, Y, strain, family, alpha, config):\n    mean, scale = X.mean(0), X.std(0)\n    scale = np.where(scale > 0, scale, 1.)\n    B = basis(X, mean, scale)\n    stress_scale = float(np.sqrt(np.mean(Y**2)))\n    yn = Y/stress_scale\n    count = 3 if family == \'exponential\' else 4\n    mask = np.tile([0., 1., 1., 1., 1., 1.], count)\n    norm = np.sqrt(Y.size)\n\n    def fun(v):\n        p = response(v, B, strain, family, reference=config[\'strain_reference\'])\n        return np.r_[(p-yn).ravel()/norm, np.sqrt(alpha)*mask*v]\n\n    def jac(v):\n        _, J = response(v, B, strain, family, reference=config[\'strain_reference\'], jacobian=True)\n        return np.vstack([J/norm, np.diag(np.sqrt(alpha)*mask)])\n\n    starts, fitted = [], []\n    for seed in range(config[\'starts\']):\n        b = np.random.default_rng(seed).normal(0, .08, (count, 6))\n        b[:3, 0] = [np.log(.2), np.log(.9), np.log([.5, 1., 2.][seed % 3])]\n        if count == 4:\n            b[3, 0] = logit(([1., 1.5, 2.][seed % 3]-.1)/7.9)\n        # Set intercept separately because the flexible family has four rows.\n        initial = b.ravel()\n        fit = least_squares(fun, initial, jac=jac, bounds=(-8, 8),\n                            max_nfev=config[\'max_nfev\'], ftol=1e-10, xtol=1e-10, gtol=1e-9)\n        fitted.append(fit)\n        starts.append(dict(seed=seed, cost=float(fit.cost), success=bool(fit.success),\n                           evaluations=fit.nfev, optimality=float(fit.optimality)))\n    best = min(fitted, key=lambda f: f.cost)\n    pbest = response(best.x, B, strain, family, reference=config[\'strain_reference\'])\n    spread = max(float(np.sqrt(np.mean((response(f.x, B, strain, family,\n                 reference=config[\'strain_reference\'])-pbest)**2))) for f in fitted)\n    sv = np.linalg.svd(jac(best.x)[:Y.size], compute_uv=False)\n    return dict(family=family, alpha=alpha, mean=mean.tolist(), scale=scale.tolist(),\n                stress_scale=stress_scale, beta=best.x.reshape(count, 6).tolist(),\n                reference=config[\'strain_reference\'], starts=starts,\n                success=bool(best.success), initialization_prediction_spread_relative_rms=spread,\n                data_jacobian_condition=float(sv[0]/max(sv[-1], 1e-300)))\n\n\ndef predict(model, X, strain):\n    return response(model[\'beta\'], basis(X, np.array(model[\'mean\']), np.array(model[\'scale\'])),\n                    strain, model[\'family\'], model[\'stress_scale\'], model[\'reference\'])\n\n\ndef metrics(Y, prediction):\n    rmse = np.sqrt(np.mean((prediction-Y)**2, axis=1))\n    nrmse = rmse/np.sqrt(np.mean(Y**2, axis=1))\n    return dict(mean_rmse=float(rmse.mean()), worst_rmse=float(rmse.max()),\n                mean_nrmse=float(nrmse.mean()), worst_nrmse=float(nrmse.max()),\n                case_rmse=rmse.tolist(), case_nrmse=nrmse.tolist())\n\n\ndef fit_shape(y, strain, family):\n    u = strain/.1\n    def fun(v, at=u):\n        n = 1. if family == \'exponential\' else v[3]\n        return v[0]+v[1]*np.exp(-v[2]*at**n)\n    count = 3 if family == \'exponential\' else 4\n    fits = []\n    for k, n in [(1., 1.), (2., 2.), (5., 3.)]:\n        initial = [max(y[-1]*.3, .01), max(y[0]-y[-1]*.3, .01), k]\n        if count == 4:\n            initial.append(n)\n        f = least_squares(lambda v: fun(v)-y, initial,\n                          bounds=([0, 0, 1e-6]+([.1] if count == 4 else []),\n                                  [1000, 1000, 100]+([8] if count == 4 else [])),\n                          x_scale=\'jac\', max_nfev=2500, ftol=1e-10, xtol=1e-10, gtol=1e-10)\n        fits.append(f)\n    best = min(fits, key=lambda f: np.sum(f.fun**2))\n    singular = np.linalg.svd(best.jac, compute_uv=False)\n    return dict(parameters=best.x.tolist(), rmse=float(np.sqrt(np.mean(best.fun**2))),\n                jacobian_condition=float(singular[0]/max(singular[-1], 1e-300)),\n                success=bool(best.success)), fun(best.x)\n\n\ndef numerical_checks():\n    strain = np.linspace(.0002, .0999, 120)\n    R, A, k, n, ref = 20., 100., 1.3, .7, .1\n    exact = lambda e: R+A*np.exp(-k*(np.asarray(e)/ref)**n)\n    derivative = lambda e: -k*n/ref*(np.asarray(e)/ref)**(n-1)*(exact(e)-R)\n    h = 1e-8\n    fd = (exact(strain+h)-exact(strain-h))/(2*h)\n    fd_error = float(np.max(np.abs(fd-derivative(strain)))/np.max(np.abs(derivative(strain))))\n    sol = solve_ivp(lambda e, s: -k*n/ref*(e/ref)**(n-1)*(s-R),\n                    (strain[0], strain[-1]), [exact(strain[0])], t_eval=strain,\n                    rtol=1e-10, atol=1e-10)\n    ode_error = float(np.max(np.abs(sol.y[0]-exact(strain)))/np.max(exact(strain)))\n    return dict(derivative_relative_error=fd_error, ode_relative_error=ode_error,\n                passed=bool(sol.success and fd_error < 1e-6 and ode_error < 1e-6))\n\n\ndef run(root, output):\n    root, output = Path(root), Path(output)\n    if output.exists():\n        raise FileExistsError(f\'Choose a new output directory: {output}\')\n    config_path = root/\'configs/effective_equation.json\'\n    config = json.loads(config_path.read_text())\n    if config[\'test_targets_permitted\']:\n        raise ValueError(\'Test target access forbidden\')\n    grid = np.linspace(config[\'strain_min\'], config[\'strain_max\'], config[\'strain_points\'])\n    ids, X, Y, audit, split = load_training(root/\'TEAM_8_extracted_results_set_1\', root/\'data/split.json\', grid)\n    folds = list(KFold(config[\'folds\'], shuffle=True, random_state=config[\'seed\']).split(ids))\n    result = dict(config=config, config_sha256=digest(config_path),\n                  code_sha256=digest(__file__), split_sha256=digest(root/\'data/split.json\'),\n                  training_ids=ids.tolist(), excluded_test_ids=split[\'test\'], raw_audit=audit,\n                  raw_history_ids_opened=[a[\'id\'] for a in audit],\n                  versions=dict(python=platform.python_version(), numpy=np.__version__, scipy=scipy.__version__),\n                  numerical_checks=numerical_checks(), families={},\n                  evaluation_status=\'Development CV: used to compare families/alpha; not unbiased final-test performance\')\n    baseline = np.empty_like(Y)\n    fold_manifest = []\n    for tr, va in folds:\n        baseline[va] = Y[tr].mean(0)\n        fold_manifest.append(dict(fit=ids[tr].tolist(), validation=ids[va].tolist()))\n    result[\'folds\'] = fold_manifest\n    result[\'mean_curve_baseline\'] = metrics(Y, baseline)\n    curves = {}\n    start = time.perf_counter()\n    for family in config[\'families\']:\n        shape, sp = [], []\n        for i, y in zip(ids, Y):\n            m, p = fit_shape(y, grid, family)\n            shape.append(dict(id=int(i), **m)); sp.append(p)\n        trials = []\n        for alpha in config[\'alphas\']:\n            oof = np.empty_like(Y); models = []\n            for fold, (tr, va) in enumerate(folds):\n                model = fit_global(X[tr], Y[tr], grid, family, alpha, config)\n                oof[va] = predict(model, X[va], grid); models.append(model)\n                print(f\'{family} alpha={alpha} fold={fold+1} done\', flush=True)\n            trials.append(dict(alpha=alpha, metrics=metrics(Y, oof), models=models, predictions=oof))\n        selected = min(trials, key=lambda t: t[\'metrics\'][\'mean_rmse\'])\n        transfer = []\n        for T, lr in sorted(set(map(tuple, X[:, :2]))):\n            va = np.where((X[:, 0] == T) & (X[:, 1] == lr))[0]\n            tr = np.setdiff1d(np.arange(len(ids)), va)\n            model = fit_global(X[tr], Y[tr], grid, family, selected[\'alpha\'], config)\n            p = predict(model, X[va], grid)\n            transfer.append(dict(temperature=T, log10_rate=lr, fit_ids=ids[tr].tolist(),\n                                 validation_ids=ids[va].tolist(), metrics=metrics(Y[va], p)))\n            print(f\'{family} condition holdout {T}/{lr} done\', flush=True)\n        model = fit_global(X, Y, grid, family, selected[\'alpha\'], config)\n        cv = selected[\'metrics\']\n        gate = dict(mean_nrmse=cv[\'mean_nrmse\'] < config[\'screening_mean_nrmse\'],\n                    worst_nrmse=cv[\'worst_nrmse\'] < config[\'screening_worst_nrmse\'],\n                    beats_mean_curve=cv[\'mean_rmse\'] < result[\'mean_curve_baseline\'][\'mean_rmse\'],\n                    selected_fits_converged=all(m[\'success\'] for m in selected[\'models\']) and model[\'success\'],\n                    numerics=result[\'numerical_checks\'][\'passed\'])\n        result[\'families\'][family] = dict(shape=shape, shape_metrics=metrics(Y, np.array(sp)),\n            trials=[{k:v for k,v in t.items() if k != \'predictions\'} for t in trials],\n            selected_alpha=selected[\'alpha\'], cv=cv, screening=gate, screening_passed=all(gate.values()),\n            condition_holdouts=transfer, final_training_model=model,\n            final_training_metrics=metrics(Y, predict(model, X, grid)))\n        curves[family] = dict(shape=np.array(sp), cv=selected[\'predictions\'], final=predict(model, X, grid))\n    result[\'elapsed_seconds\'] = time.perf_counter()-start\n    result[\'qualified_families\'] = [f for f,r in result[\'families\'].items() if r[\'screening_passed\']]\n    result[\'decision\'] = \'screening_passed_not_physical_validation\' if result[\'qualified_families\'] else \'not_qualified_do_not_create_pinn_yet\'\n    output.mkdir(parents=True)\n    (output/\'results.json\').write_text(json.dumps(result, indent=2, allow_nan=False)+\'\\n\')\n    with (output/\'predictions.csv\').open(\'w\', newline=\'\') as f:\n        w = csv.writer(f); w.writerow([\'id\', \'E11\', \'observed_Svm\', \'mean_curve_cv\']+\n                                      [f\'{family}_{kind}\' for family in curves for kind in curves[family]])\n        for i, case in enumerate(ids):\n            for j, strain in enumerate(grid):\n                w.writerow([case, strain, Y[i,j], baseline[i,j]]+\n                           [curves[family][kind][i,j] for family in curves for kind in curves[family]])\n    cache = root/\'work\'/\'matplotlib\'\n    cache.mkdir(parents=True, exist_ok=True)\n    os.environ.setdefault(\'MPLCONFIGDIR\', str(cache))\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    fig, axes = plt.subplots(2, 3, figsize=(12, 7), constrained_layout=True)\n    for ax, case in zip(axes.flat, [2, 18, 40, 52, 60, 64]):\n        i = list(ids).index(case)\n        ax.plot(grid, Y[i], color=\'black\', label=\'Supplied curve\')\n        for family, style in [(\'exponential\', \'--\'), (\'power_exponential\', \'-\')]:\n            ax.plot(grid, curves[family][\'cv\'][i], style, label=f\'{family} case-CV\')\n        ax.set(title=f\'Training case {case}; T={X[i,0]:g}, log-rate={X[i,1]:g}\',\n               xlabel=\'E11 (source strain coordinate)\', ylabel=\'Svm (source stress units)\')\n    axes.flat[0].legend(fontsize=7)\n    fig.suptitle(\'Effective-equation predictions: cases held out within the 36 training cases\')\n    fig.savefig(output/\'case_validation.png\', dpi=150); plt.close(fig)\n    print(json.dumps(dict(decision=result[\'decision\'],\n        families={f: dict(mean_rmse=r[\'cv\'][\'mean_rmse\'], mean_nrmse=r[\'cv\'][\'mean_nrmse\'],\n                          worst_nrmse=r[\'cv\'][\'worst_nrmse\'], gate=r[\'screening\'])\n                  for f,r in result[\'families\'].items()})), flush=True)\n    return result\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--root\', type=Path, default=ROOT)\n    parser.add_argument(\'--output\', type=Path, required=True)\n    args = parser.parse_args()\n    run(args.root, args.output)\n'
assert hashlib.sha256(source.encode()).hexdigest() == '9564611cd5895f5647a5f1af0e5807ac0724f1e6fd8364857e7e4f104a5c928a'
module_dir = LOCAL_ROOT/'src'
module_dir.mkdir(exist_ok=True)
(module_dir/'effective_equation.py').write_text(source)
sys.path.insert(0, str(module_dir))
import effective_equation as equation


In [ ]:
split = json.loads((LOCAL_ROOT/'data/split.json').read_text())
raw = LOCAL_ROOT/'TEAM_8_extracted_results_set_1'
members = ['simulation_matrix.csv'] + [f'simulation_{i:04d}/STR_STR.OUT' for i in split['train']]
with zipfile.ZipFile(ZIP_SOURCE) as archive:
    names = archive.namelist()
    for relative in members:
        matches = [name for name in names if name.replace('\\','/').rstrip('/').endswith('/'+relative)
                   or name.replace('\\','/') == relative]
        if len(matches) != 1:
            raise ValueError(f'Expected one archive member for {relative}: {matches}')
        destination = raw/relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(archive.read(matches[0]))
assert sorted(int(p.name.split('_')[1]) for p in raw.glob('simulation_*') if p.is_dir()) == sorted(split['train'])
print('Extracted matrix and exactly 36 training histories. No test history extracted.')


In [ ]:
result = equation.run(LOCAL_ROOT, OUTPUT)
assert set(result['raw_history_ids_opened']) == set(split['train'])
assert not set(result['raw_history_ids_opened']) & set(split['test'])
print('Decision:', result['decision'])
print('Saved to:', OUTPUT)


In [ ]:
from IPython.display import display, Image
for family, record in result['families'].items():
    scores = record['cv']
    print(f"{family}: mean normalized RMSE {100*scores['mean_nrmse']:.2f}%; worst {100*scores['worst_nrmse']:.2f}%")
    print('Screening:', record['screening'])
display(Image(filename=str(OUTPUT/'case_validation.png')))


## Interpret the decision

A passing fit would support only an effective equation within the observed loading
family and E11 interval 0.0002–0.0999. It would not identify the original constitutive
law, establish experimental validity, or demonstrate a porosity effect.

The local qualification failed both candidates. In particular, case 60 is the only
training case at matrix temperature 200 and rate 1e-5; the remaining data did not
predict it reliably. Retain it and report the limitation. If a repeat run also fails,
continue equation research before constructing the MLP/PINN comparison.

The differential residual is
`dSvm/dE11 + (k*n/0.1)*(E11/0.1)^(n-1)*(Svm-R) = 0`.
An initial value at E11=0.0002 must come from the condition-dependent equation,
never from a validation/test case's measured stress. E11 is not renamed plastic strain.
